> **How to run**: This notebook reads `train.csv`, `val.csv`, and `test.csv` from the current folder. Copy the three CSVs from the provided `inputs/` folder to the same location as the notebook and run it from top to bottom. The neural network sweep runs for up to 600 epochs, so it may take a while. Results from a previous training run are preserved in the submission report and in the `results/` folder.

# Homework 2 · Problem 3(a)–(d)
## From a 12-year-old seeing this for the first time to a math/ML expert: complete study notes on RKS and MLP

**Learning goals**: First build the intuition of predicting a height from a 2D position, then understand regression, MSE, matrices, sine/cosine, random features, the ridge normal equations, ReLU, backpropagation, Adam, and early stopping. At the end, we run 3(a)–(d) from start to finish on **the actual `train.csv`, `val.csv`, and `test.csv` provided by the professor**.

### Actual files used

| Data | Rows | Inputs | Target |
|---|---:|---|---|
| Train | 600 | `x1`, `x2` | `y` present |
| Validation | 200 | `x1`, `x2` | `y` present |
| Test | 200 | `x1`, `x2` | **no `y`** |

**How to read**: Proceed in the order `explanation → code cell right below → check the actual result`. Every statement carries a comment explaining what it means. The code is not a set of independent snippets but **one complete program that runs from top to bottom**.

### Setup

1. Put this `.ipynb` and `train.csv`, `val.csv`, `test.csv` in the same folder. (In the repository, copy the three files from `data/` into this folder.)
2. Install: `python -m pip install numpy pandas matplotlib torch jupyter`
3. Launch Jupyter from that folder and click **Kernel → Restart & Run All**.
4. All figures and experiment tables are written to `problem3_outputs/`, and the two prediction files for submission are written to the current folder.

---

# Part A. A starting point for a friend who knows nothing yet

## 1. What is a triple of numbers `(x1, x2, y)`?

Picture the map of a miniature amusement park. Think of `x1` as the horizontal position, `x2` as the vertical position, and `y` as the height of the ground at that spot. In this assignment `x1,x2` are not real map coordinates, but the structure — **predicting one number from two inputs** — is exactly the same.

| x1 | x2 | y | Meaning |
|---:|---:|---:|---|
| 0 | 0 | 1 | Height 1 at the center |
| 1 | 0 | 3 | Moving right, the height is 3 |
| 0 | 1 | 2 | Moving up, the height is 2 |

The learning goal is to obtain a `function f(x1,x2) that tells you the height when you tell it where you are`.

### Data rows and input columns

- One **row** = one observation. It is a single point on the map.
- **Columns** `x1`, `x2` = the 2 inputs; `y` = the 1 target we must predict.
- **Model** = a rule that looks at the inputs and computes a prediction `ŷ`.

`ŷ` is read "y-hat" and means **the model's estimate, not the true answer**.

## 2. How do we turn "predicted well" into a number? — MSE

If the true values are `[1, 3, 5]` and the model predicts `[1, 5, 4]`, the individual errors `(prediction − truth)` are `[0, 2, -1]`.

**If we simply add up the errors**, large positive and negative errors cancel out. So we square the errors to get `[0, 4, 1]` and then take the average.

\[
\mathrm{MSE}=\frac{0+4+1}{3}=\frac53\approx1.667.
\]

In general, with `n` observations we write

\[
\boxed{\mathrm{MSE}=\frac1n\sum_{i=1}^{n}(y_i-\hat y_i)^2}
\]

A smaller MSE means better predictions, but it gives **a larger penalty to a single point with a large error**.

## 3. Why do we need three data splits?

**Train (practice problems)** is used only to learn the model's coefficients. **Validation (mock exam)** is used to choose the best among several models/hyperparameters. **Test (final exam)** is used to produce predictions with the chosen model. The test set in this assignment has no answer key `y` at all, so the test MSE cannot be computed.

"We chose by looking at validation" means we have already tried several candidates against the validation score. Therefore the validation MSE is a performance estimate used for model selection, not the still-unknown final exam score.

---

# Part B. From zero math to RKS

## 4. Why is a single straight line not enough?

A very simple prediction function is

\[
\hat y=b_0+b_1x_1+b_2x_2
\]

Here `b0` is the base height, `b1` is the change when you move one step to the right, and `b2` is the change when you move one step up. The graph of this function is a **flat inclined plane**. But real terrain curves like mountains, valleys, and waves. The idea we use in that case is **feature transformation**.

## 5. `sin` and `cos`, which make a single wave

Sine and cosine are wave functions that repeat up and down. Given an angle `t`, both `sin(t)` and `cos(t)` return a value between `-1` and `1`. Here we use **radians**. Moving by about `2π` completes one period.

A wave that takes a single number as input can also be applied to a 2D position:

\[
\phi_j^{\sin}(x)=\sin(w_j^\top x+b_j),\quad
\phi_j^{\cos}(x)=\cos(w_j^\top x+b_j).
\]

Taking every symbol apart:

- `x=[x1,x2]`: the input position on the map.
- `w_j=[w_{j1},w_{j2}]`: two numbers that set **in which direction and how fast the wave changes**.
- `w_j^T x = w_{j1} x1+w_{j2} x2`: the dot product, which multiplies the two pairs of values and adds them.
- `b_j`: the **phase**, which shifts the wave back and forth.
- `φ`: the newly created number, i.e., a **feature**.

### Computing it by hand

Let `x=(1,2), w=(0.5,-1), b=0.2`.

\[
w^\top x+b=(0.5)(1)+(-1)(2)+0.2=-1.3
\]

So we get **two features**: `sin(-1.3) ≈ -0.9636` and `cos(-1.3) ≈ 0.2675`.

## 6. Why is this called Random Kitchen Sinks?

We do not search for the `w`, `b` that build the features. Instead we first **draw them at random from fixed probability distributions and then freeze those values**.

\[
\boxed{w_j\sim\mathcal N(0,I_2/\sigma^2),\quad b_j\sim U[0,2\pi]}
\]

- `𝒩` is the normal distribution. Each coordinate of `w` comes from a normal distribution with mean 0 and **standard deviation `1/σ`**.
- `U[0,2π]` is the uniform distribution, which draws any point between 0 and `2π` with equal likelihood.
- **When `σ` is small**, `w` tends to be larger, so the waves change faster. **When `σ` is large**, slowly varying waves are more likely.

Each `w,b` yields **two numbers**, `sin` and `cos`, so `h` pairs give `D=2h` features in total. Example: `D=20` means sin/cos pairs for 10 different waves.

\[
\phi(x)=[\sin(w_1^\top x+b_1),\cos(w_1^\top x+b_1),\ldots,\sin(w_h^\top x+b_h),\cos(w_h^\top x+b_h)].
\]

## 7. Why does linear regression on random waves give a nonlinear prediction?

The final function RKS produces is

\[
\hat y(x)=\beta_0+\sum_{k=1}^{D}\beta_k\phi_k(x).
\]

`β` is found by learning, but the shapes of `φ` are set at random and never change afterwards. Because `φ` consists of sines/cosines, even a simple weighted sum with the `β`s becomes a **winding nonlinear function** of the original coordinates `x1,x2`.

- **Fixed**: the random frequencies `W` and random phases `b`.
- **Learned**: `β`, which decides how much of each feature above to mix in.

## 8. Why do we need Ridge on top of that?

If we build `D=320` features, linear regression has **321** coefficients including the intercept. That is enough to memorize 600 points, so we penalize large coefficients.

\[
J(\beta)=\underbrace{\|A\beta-y\|_2^2}_{\text{sum of squared prediction errors}}+
\lambda\underbrace{\beta^\top P\beta}_{\text{sum of squared coefficients, excluding intercept}},
\quad P=\operatorname{diag}(0,1,\ldots,1).
\]

Here `A=[1, φ(x)]` is an `(n,D+1)` matrix when there are `n` data points. The `1` in the first column is the slot for the intercept `β0`. The coefficient `β` is a vector of length `D+1`.

### Why does this become a normal equation?

The vector `r=Aβ−y` holds each data point's prediction error. Differentiating `rᵀr` gives `2Aᵀ(Aβ−y)`, and differentiating the ridge penalty gives `2λPβ`.

\[
\nabla J=2A^\top(A\beta-y)+2\lambda P\beta=0
\]

Expanding and rearranging,

\[
2A^\top A\beta-2A^\top y+2\lambda P\beta=0
\implies \boxed{(A^\top A+\lambda P)\beta=A^\top y}.
\]

The `A.T @ A`, `penalty`, and `np.linalg.solve(left,right)` in the code are **exactly this formula**. **Important**: the first term of this loss is not the MSE but the *sum of squared errors (SSE)*. The MSE used for evaluation is that value divided by `n`. The relative strength of the penalty depends on this definition.

### Expert connection — the Gaussian RBF kernel

Using the sine/cosine addition formulas, the inner product of one pair is

\[
\sin(a)\sin(c)+\cos(a)\cos(c)=\cos(a-c).
\]

Therefore, for \(h\) pairs of features, \(h^{-1}\phi(x)^\top\phi(x')\) approaches, for large `h`,

\[
\mathbb E_w[\cos(w^\top(x-x'))]=
\exp\!\left(-\frac{\|x-x'\|^2}{2\sigma^2}\right)
\]

This is exactly the Gaussian RBF kernel. **Note**: the `φ` in the assignment implementation is the *unnormalized* sin/cos feature map specified in the problem; the `1/h` factor is applied only for the RBF-approximation comparison. The ridge `λ` in the code must therefore be interpreted relative to that scale.

---

# Part C. From zero math to MLP

## 9. What is a single neuron?

A single neuron takes two inputs, forms the weighted sum

\[
z=w_1x_1+w_2x_2+b
\]

and passes it through `ReLU(z)=max(0,z)`. Negative values become 0 and positive values pass through unchanged. This lets us represent **bends across multiple regions** that straight lines alone could not.

A network with **exactly one hidden layer** of `H` hidden nodes can be written as

\[
\boxed{\hat y(x)=c+\sum_{j=1}^{H}v_j\operatorname{ReLU}(w_j^\top x+b_j)}
\]

It is wired as 2 inputs → H hidden nodes → 1 output. Unlike RKS, here **all of** `w_j`, `b_j`, `v_j`, `c` are learned.

## 10. How does learning happen? — a small numeric example

Take the very simple model `ŷ=w·x` with `x=2`, true `y=5`, and initial `w=1`.

- The first prediction is `ŷ=1·2=2`.
- The squared error is `(2−5)²=9`.
- The derivative is `d(w x−y)²/dw = 2(wx−y)x = 2(2−5)·2=−12`.
- With learning rate `η=0.1`, the new coefficient is `w_new = 1−0.1(−12)=2.2`.
- The next prediction is `ŷ=2.2·2=4.4`; the squared error is `(4.4−5)²=0.36`.

In other words, **we use the derivative to find out in which direction to adjust so the error decreases**, and update a little at a time. This is the core of gradient descent.

### Why does `loss.backward()` work for an MLP?

The output passes through several computations: Linear → ReLU → Linear. Applying the chain rule, we can compute the gradients in sequence: `final layer that changes the prediction → middle layer → first layer`. PyTorch's automatic differentiation (Autograd) carries out this process.

`optimizer.step()` uses the computed gradients to actually change the weight values. Here we use **Adam** instead of plain SGD. Adam uses moving averages of the gradient and of the squared gradient to adapt the update size for each parameter.

## 11. What are epochs, batches, and early stopping?

- **Epoch** = one full pass over all 600 rows of training data.
- **Batch** = a small chunk read at once. With `batch_size=64`, the 600 rows are split into about 10 updates.
- **Learning rate** = a number that controls the size of each update.
- **Validation MSE** = a score that tracks how well the model does *on new data*.
- **Early stopping** = stop training when validation fails to improve for a number of consecutive epochs, and restore the weights from the best epoch.

The validation rows are **never used directly for weight gradients/updates**. Validation values are used only for model selection and early stopping. This distinction is an important safeguard against data leakage.

## 12. RKS vs MLP, one sentence each

| Aspect | RKS + Ridge | 1-hidden-layer MLP |
|---|---|---|
| Feature functions | Drawn at random and **fixed** | **Learned** to fit the data |
| Coefficients combining the output | Learned via the ridge linear system | Learned via backpropagation + Adam |
| Shape of prediction | Smooth combination of sin/cos | Piecewise-linear combination built from ReLUs |
| Hyperparameter sweep | Total number of features `D` | Number of hidden nodes `H` |
| Guard against overfitting | Ridge L2 penalty | Validation-based early stopping |

**We do not conclude that a model is good merely because its training error is low.** The final selection criterion is the validation MSE.

---

# Part D. The complete Python code for submission — line-by-line commentary

Run every code cell below **from top to bottom**. Each statement carries a comment explaining why it is needed. Read the code while comparing how the mathematical symbols are implemented against the concept explanations above.

### Bonus: check MSE and sine waves with actual numbers

This cell is a short exercise separate from the submission code and does not clash with any later variables.

In [ ]:
# Number practice for a 12-year-old friend: square the errors and take the average
small_true = np.array([1, 3, 5]) if "np" in globals() else [1, 3, 5]  # If NumPy has not been imported yet, use a plain list.
small_pred = [1, 5, 4]  # Values we pretend the model predicted.
squared_errors = [(a-b)**2 for a, b in zip(small_true, small_pred)]  # Square each error.
print("Squared errors:", squared_errors, "MSE:", sum(squared_errors)/len(squared_errors))  # Confirm that MSE = 5/3.

### Note · same code as the complete Python file

From here on we use the **entire** code of the `.py` file, in order. Read the explanation attached to each cell and run it.

In [ ]:
# -*- coding: utf-8 -*-
"""Homework 2 · Problem 3(a)–(d): RKS + Ridge vs one-hidden-layer MLP.

How to run (with the files in the same folder): python problem3_rks_mlp.py  (from the homework_2 folder: python run_all.py --problem 3)
Required files: train.csv, val.csv, test.csv
Dependencies: numpy, pandas, matplotlib, torch
Next to every statement there is a comment explaining 'what this line does'.
Important: test.csv has no ground-truth y, so we do not compute a test MSE.
"""

### Code 0. What is `import`?

It is like getting your pencils and textbooks ready before starting at a new school. `numpy` is a matrix calculator, `pandas` is a spreadsheet tool, `matplotlib` is a drawing tool, `torch` and `nn` are tools for building neural networks, and `DataLoader` is a tool that feeds data in small chunks. `matplotlib.use("Agg")` lets us save PNGs even on a server without a display. Without `from torch import nn`, `nn.Module` gets underlined (undefined-name error).


In [ ]:
# %% 0. Libraries: first define every name we will need.
import copy  # Safely copies the best neural-network weights during training.
from pathlib import Path  # Handles CSV and output file paths independently of the OS.
import numpy as np  # Provides arrays, matrix multiplication, random numbers, and linear-system solvers.
import pandas as pd  # Reads CSV files and stores experiment results as tables.
import matplotlib  # Matplotlib's base package, used to save figures to files.
matplotlib.use("Agg")  # Configure it to save PNG images even in an environment without a display.
import matplotlib.pyplot as plt  # Draws scatter plots, loss curves, and contour surfaces.
import torch  # PyTorch, which computes neural networks and does automatic differentiation.
from torch import nn  # Gives us nn.Module, nn.Linear, nn.ReLU, nn.MSELoss, etc.
from torch.utils.data import DataLoader, TensorDataset  # Feeds the training data in mini-batches.

### Code 1. Where do `SEED`, `DEVICE`, `D`, and `λ` come from?

**Every constant that appears from now on is first defined in this cell.** `SEED=42` is for experiment reproducibility, and `DEVICE` means CUDA if a GPU is available, otherwise CPU. `D=20` means 20 features = 10 sin/cos pairs, `RKS_SIGMA=1.0` sets the wavelength, and `RKS_LAMBDA=0.01` is the strength of the coefficient penalty. The MLP uses `H∈{16,32,64,128}`, learning rate `0.002`, at most 600 epochs, and early stopping after 45 epochs. **None of these settings is used anywhere later before being defined.**


In [ ]:
# %% 1. Fixed settings of the experiment: changing these variables affects the whole experiment.
SEED = 42  # A number that improves reproducibility when the same program is run again.
np.random.seed(SEED)  # Fix NumPy's global random-number-generator state.
torch.manual_seed(SEED)  # Fix PyTorch's random state (e.g., for weight initialization).
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Use a CUDA GPU if available, otherwise the CPU.
DATA_DIR = Path(".")  # Look for train.csv, val.csv, test.csv in the current working folder.
OUTPUT_DIR = Path("problem3_outputs")  # Name of the folder where figures and experiment tables are saved.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # Create the output folder if missing; skip if it already exists.
RKS_D_VALUES = [20, 40, 80, 160, 320]  # Candidate values for the total number of RKS sin/cos features D=2h.
RKS_SIGMA = 1.0  # Scale that sets the standard deviation of the random frequencies to 1/sigma.
RKS_LAMBDA = 0.01  # Strength of the ridge L2 penalty added to the RKS linear-regression coefficients.
MLP_WIDTH_VALUES = [16, 32, 64, 128]  # Candidate numbers of hidden nodes for the MLP.
MLP_LR = 0.002  # Learning rate Adam uses when updating the weights.
MLP_BATCH_SIZE = 64  # Number of samples used for one weight update.
MLP_MAX_EPOCHS = 600  # Maximum number of passes the MLP makes over the full training data.
MLP_PATIENCE = 45  # Stop once this many epochs pass with no improvement in validation MSE.
print(f"Compute device: {DEVICE}")  # Confirm in the terminal which CPU/GPU was actually selected.

### Code 2. When we load a CSV, what shape does it take in memory?

`train_df` is a `pandas.DataFrame`, and `X_train` is a NumPy matrix containing only numbers. The actual shapes are `X_train.shape=(600,2)` and `y_train.shape=(600,)`. Likewise, `X_val=(200,2)`, `y_val=(200,)`, `X_test=(200,2)`. `assert` is a safety device that makes the program **stop immediately instead of silently producing a wrong answer** when one of its assumptions is violated. **The test set has no y.**


In [ ]:
# %% 2. Read the three given CSVs and check that their shapes and contents are valid.
train_df = pd.read_csv(DATA_DIR / "train.csv")  # Read the 600 rows of training data that have ground-truth y.
val_df = pd.read_csv(DATA_DIR / "val.csv")  # Read the 200 rows of validation data that have ground-truth y.
test_df = pd.read_csv(DATA_DIR / "test.csv")  # Read the 200 rows of test data that have no ground-truth y.
assert list(train_df.columns) == ["x1", "x2", "y"], "train.csv columns must be x1,x2,y"  # Check the training column names and order.
assert list(val_df.columns) == ["x1", "x2", "y"], "val.csv columns must be x1,x2,y"  # Check the validation column names and order.
assert list(test_df.columns) == ["x1", "x2"], "test.csv columns must be x1,x2"  # The test set must not have a y column.
X_train = train_df[["x1", "x2"]].to_numpy(dtype=np.float64)  # Turn the two training input columns into an (n_train,2) array.
y_train = train_df["y"].to_numpy(dtype=np.float64)  # Turn the training targets into a vector of length n_train.
X_val = val_df[["x1", "x2"]].to_numpy(dtype=np.float64)  # Turn the two validation input columns into an (n_val,2) array.
y_val = val_df["y"].to_numpy(dtype=np.float64)  # Turn the validation targets into a vector of length n_val.
X_test = test_df[["x1", "x2"]].to_numpy(dtype=np.float64)  # For the test set we use only the inputs x1,x2.
for array in (X_train, y_train, X_val, y_val, X_test):  # Inspect every data array in turn.
    assert np.isfinite(array).all(), "The data contains NaN/infinity"  # Check for numbers that cannot be computed with.
assert X_train.ndim == 2 and X_train.shape[1] == 2  # Confirm the training input dimension is exactly 2.
assert X_val.ndim == 2 and X_val.shape[1] == 2  # Confirm the validation input dimension is exactly 2.
assert X_test.ndim == 2 and X_test.shape[1] == 2  # Confirm the test input dimension is exactly 2.
print(f"Data: train={len(X_train)} / val={len(X_val)} / test={len(X_test)}")  # Show the number of observations in each split.

### Code 3. Mapping the MSE formula to the code, piece by piece

In the formula `MSE=(1/n)Σ(y_i−ŷ_i)²`, `y_true - y_pred` is the error vector, `** 2` squares each element, and `np.mean(...)` takes the average. `.reshape(-1)` prevents the *broadcasting bug* in which a shape mismatch between `(n,1)` and `(n,)` makes the matrix blow up unexpectedly.


In [ ]:
# %% 3. Define the mathematical MSE directly in code.
def mse(y_true, y_pred):  # Takes an array of true values and an array of predictions and computes a single MSE.
    y_true = np.asarray(y_true).reshape(-1)  # Turn the true values into a 1-D array of shape (n,).
    y_pred = np.asarray(y_pred).reshape(-1)  # Normalize the predictions to shape (n,) as well.
    assert y_true.shape == y_pred.shape  # Check the lengths so that matching rows can be compared.
    return float(np.mean((y_true - y_pred) ** 2))  # Return the average of (true − predicted)^2.

### Code 4. (a) Why do we need `scatter` and `colorbar`?

For a single point, the coordinate `X_train[i,0]` is `x1` and `X_train[i,1]` is `x2`. The point's color is set by `y_train[i]`. The two inputs are shown as the point's position and the single target as its color, so 3-dimensional information fits into a 2D figure. `xlim` and `ylim` use the `[-3,3]` range given in the problem. The figure file `3a_training_scatter.png` is produced.


In [ ]:
# %% 4. [3(a)] Plot the training data as points on the plane and show the target y as color.
fig, ax = plt.subplots(figsize=(7, 5.8))  # Create the figure and axes to be saved as a PNG.
scatter = ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="viridis", s=28, alpha=0.85)  # Position = (x1,x2), point color = y.
ax.set_xlabel("x1")  # Label the horizontal axis with the name of the first input.
ax.set_ylabel("x2")  # Label the vertical axis with the name of the second input.
ax.set_title("3(a) Training points colored by y")  # Put the sub-question and the meaning of the plot in the title.
ax.set_xlim(-3, 3)  # Use the x1 range given in the problem.
ax.set_ylim(-3, 3)  # Use the x2 range given in the problem.
ax.set_aspect("equal", adjustable="box")  # Use the same distance scale in both directions so the spatial pattern is not distorted.
fig.colorbar(scatter, ax=ax, label="observed y")  # Add a legend showing which y value each color represents.
fig.tight_layout()  # Adjust margins so text and plot are not cut off.
fig.savefig(OUTPUT_DIR / "3a_training_scatter.png", dpi=180)  # Save the 3(a) figure for submission.
plt.close(fig)  # Free the memory of the figure we no longer need.

### Code 5. (b) RKS has four parts

1. **`__init__`**: Stores the settings D, σ, λ and draws the random `W` and `b` once. `W.shape=(2,h)`, `b.shape=(h,)`. They never change during training.
2. **`_features`**: Takes `X.shape=(n,2)`; computing `X @ W` gives `(n,h)`. `+ b` is **broadcasting**, which adds the same phase vector to every row. Then it fills the even columns with sin and the odd columns with cos to build `phi.shape=(n,D)`.
3. **`_design`**: Prepends a column of ones for the intercept, giving `A.shape=(n,D+1)`.
4. **`fit/predict`**: Solves the linear system with `AᵀA+λP` and stores `β`; for new data it computes `A_new @ β`.

In this model **only β is learned**; `W,b` are merely generated at random. This is the decisive difference from the MLP. Because the `seed` is fixed at initialization, the random waves in the common leading part are kept even as the number of features grows with D.


In [ ]:
# %% 5. [3(b)] RKS model: implement random sin/cos features + ridge regression from scratch.
class RandomKitchenSinks:  # Bundles creation, transformation, training, and prediction of the RKS model into one class.
    def __init__(self, D=80, sigma=1.0, ridge_lambda=0.01, seed=42):  # Takes the hyperparameters to set before training.
        if D <= 0 or D % 2 != 0:  # sin/cos come in pairs, so D must be a positive even number.
            raise ValueError("D must be a positive even integer")  # Report an invalid D immediately.
        if sigma <= 0 or ridge_lambda <= 0:  # Require both σ and λ to be positive.
            raise ValueError("sigma and ridge_lambda must be positive")  # Filter out impossible settings.
        self.D = int(D)  # Store D, the total number of transformed features.
        self.h = self.D // 2  # The number of sin/cos pairs h is D/2.
        self.sigma = float(sigma)  # Store the scale of the random frequencies.
        self.ridge_lambda = float(ridge_lambda)  # Store the ridge regularization strength.
        rng_w = np.random.default_rng(seed)  # Independent random generator for drawing the random frequencies w.
        rng_b = np.random.default_rng(seed + 1)  # Separate random generator for drawing the random phases b.
        self.W = rng_w.normal(0.0, 1.0 / self.sigma, size=(self.h, 2)).T  # Draw both coordinates of w_j from N(0,1/sigma^2) and arrange as (2,h).
        self.b = rng_b.uniform(0.0, 2.0 * np.pi, size=self.h)  # Draw each sine wave's starting offset b_j from [0,2π].
        self.coef_ = None  # Not trained yet, so there are no final linear-regression coefficients.

    def _features(self, X):  # Turns the inputs x1,x2 into many sin/cos numbers.
        angle = X @ self.W + self.b  # Compute w_j^T x_i + b_j for all data; the result is (n,h).
        phi = np.empty((len(X), self.D), dtype=np.float64)  # Create an empty (n,D) matrix to hold the output sin/cos features.
        phi[:, 0::2] = np.sin(angle)  # Fill every even column (0,2,4,...) with sine values.
        phi[:, 1::2] = np.cos(angle)  # Fill every odd column (1,3,5,...) with cosine values.
        return phi  # Return a matrix with D features per data point.

    def _design(self, X):  # Builds the linear-regression design matrix including the intercept b0.
        ones = np.ones((len(X), 1), dtype=np.float64)  # Create the column of '1's that multiplies the intercept coefficient.
        return np.column_stack((ones, self._features(X)))  # Concatenate the leading column of 1s + D features and return (n,D+1).

    def fit(self, X, y):  # Learns the ridge coefficients from the given inputs and targets.
        A = self._design(X)  # Build the actual design matrix A for all data.
        penalty = np.eye(self.D + 1, dtype=np.float64)  # Create a (D+1)×(D+1) identity matrix for the L2 penalty.
        penalty[0, 0] = 0.0  # The first coefficient is the intercept, so exempt it from the ridge penalty.
        left = A.T @ A + self.ridge_lambda * penalty  # Compute the left-hand side of the normal equation, A^T A + λP.
        right = A.T @ y  # Compute the right-hand side of the normal equation, A^T y.
        self.coef_ = np.linalg.solve(left, right)  # Find the coefficients by solving the linear system instead of computing an explicit inverse.
        return self  # Return the trained object itself so it can be used like .fit(...).predict(...).

    def predict(self, X):  # Predicts y for new data using the learned coefficients.
        if self.coef_ is None:  # Check whether this model has never been fit.
            raise RuntimeError("Call fit() before predict()")  # Raise a clear error if predicting before training.
        return self._design(X) @ self.coef_  # Multiply the new design matrix by the coefficient vector and return the predictions.

### Code 6. What kind of experiment is a hyperparameter sweep?

We vary `D=20,40,80,160,320` while keeping `σ`, `λ`, the training data, and the random seed fixed. **Only one condition should change at a time**, so that we can interpret how D affects performance. For each model we record **both** the train MSE and the validation MSE. The best model is the D with the lowest `val_mse`. Choosing with the validation y here is allowed, but test y does not even exist and must not be used.


In [ ]:
# %% 6. [3(b)] Try different D values and record both the train MSE and the validation MSE.
rks_rows = []  # Empty list to hold the evaluation records of the RKS candidates.
for D in RKS_D_VALUES:  # Try the predefined D=20,40,80,160,320 in order.
    candidate = RandomKitchenSinks(D=D, sigma=RKS_SIGMA, ridge_lambda=RKS_LAMBDA, seed=SEED)  # Candidate model with every condition except D held fixed.
    candidate.fit(X_train, y_train)  # Learn each candidate's ridge coefficients using only the training data.
    train_mse = mse(y_train, candidate.predict(X_train))  # Record the prediction error on the training data.
    val_mse = mse(y_val, candidate.predict(X_val))  # Record the prediction error on the unseen validation data.
    rks_rows.append({"D": D, "train_mse": train_mse, "val_mse": val_mse})  # Store D and the two losses as one record.
    print(f"RKS D={D:3d} | train MSE={train_mse:.6f} | val MSE={val_mse:.6f}")  # Print each candidate's performance immediately.
rks_table = pd.DataFrame(rks_rows)  # Turn the records above into a table that can be saved as CSV.
rks_table.to_csv(OUTPUT_DIR / "3b_rks_sweep.csv", index=False)  # Save the loss for each D as an experiment data file.
best_rks_idx = int(rks_table["val_mse"].idxmin())  # Find the row index with the minimum validation MSE.
best_D = int(rks_table.loc[best_rks_idx, "D"])  # Select the optimal D recorded in that row.
print(f"Selected best RKS D: {best_D}")  # Show the setting selected by the validation criterion.

### Code 7. What "two figures" means in the assignment wording

The professor asked for `Train MSE vs D` and `Validation MSE vs D` from the sweep as **two separate figures**. `plot_sweep` wraps the figure-saving procedure in a function so the same code can be reused. In `matplotlib.pyplot`'s `plot(x,y)`, x is D and y is the MSE. The output file names include the assignment problem number.


In [ ]:
# %% 7. Draw the hyperparameter sweep plots as two separate figures for train and validation.
def plot_sweep(table, x_name, y_name, title, filename):  # Reusable function that saves one loss column of the table as a line plot.
    fig, ax = plt.subplots(figsize=(6.5, 4.5))  # Create the figure and axes.
    ax.plot(table[x_name], table[y_name], marker="o", linewidth=2)  # Plot horizontal axis = hyperparameter, vertical axis = MSE.
    ax.set_xlabel(x_name)  # Write D or hidden_width on the horizontal axis.
    ax.set_ylabel("Mean squared error (MSE)")  # Write the evaluation metric on the vertical axis.
    ax.set_title(title)  # Add a title saying what the plot compares.
    ax.grid(True, alpha=0.3)  # Draw a light grid so values are easy to compare by eye.
    fig.tight_layout()  # Adjust margins so text is not cut off.
    fig.savefig(OUTPUT_DIR / filename, dpi=180)  # Save as a PNG image.
    plt.close(fig)  # Free the memory.

plot_sweep(rks_table, "D", "train_mse", "3(b) RKS training MSE vs D", "3b_rks_training_mse.png")  # The required RKS training-loss plot.
plot_sweep(rks_table, "D", "val_mse", "3(b) RKS validation MSE vs D", "3b_rks_validation_mse.png")  # The required RKS validation-loss plot.

### Code 8. Why does the MLP class inherit from `nn.Module`?

`nn.Module` is the standard "parent" of PyTorch neural networks. Only after calling `super().__init__()` does PyTorch register the **learnable matrices and biases** of `nn.Linear`, so that `model.parameters()` and `.to(DEVICE)` work. `Linear(2,width)` turns the two inputs into width weighted sums, `ReLU` sets negative parts to 0, and `Linear(width,1)` turns them into a single final number. **ReLU is an activation function, not a hidden layer**, so this MLP has exactly one hidden layer.


In [ ]:
# %% 8. [3(c)] Write an MLP class with exactly one hidden layer.
class OneHiddenMLP(nn.Module):  # Use nn.Module, which every PyTorch neural network inherits from, as the parent class.
    def __init__(self, width):  # width is the number of hidden nodes in the middle layer.
        super().__init__()  # Run the parent constructor so PyTorch properly tracks the internal weights.
        self.net = nn.Sequential(  # Connect the layers in the listed order to form one network.
            nn.Linear(2, width),  # Turn the two numbers (x1,x2) into width weighted sums.
            nn.ReLU(),  # Set negatives to 0 and pass positives unchanged, adding nonlinearity.
            nn.Linear(width, 1),  # Turn the width hidden activations into a single final prediction.
        )  # End of the Sequential definition.

    def forward(self, x):  # Defines the order of computation when an input x enters the model.
        return self.net(x).squeeze(1)  # Turn the (batch,1) output into (batch,) to match the target shape.

### Code 9. The execution order that really matters in the training function

`x_train_tensor` and `y_train_tensor` convert the NumPy arrays into tensors. `TensorDataset` pairs x and y row by row, and `DataLoader` serves them shuffled in chunks of 64. Without `.to(DEVICE)` on the model and the batches, they may end up on different devices (CPU/GPU) and cause an error.

**5 steps for every batch**: clear old gradients with `zero_grad()` → predict with `model(batch_x)` → measure the error with `loss_fn` → differentiate with `backward()` → update the weights with `step()`.

**4 steps for every epoch**: switch to evaluation mode with `eval()` → compute the full train/val MSE under `no_grad()` → if it is the best validation score so far, save the weights with `deepcopy(model.state_dict())` → if there is no improvement, increment a counter and stop after 45 in a row. The final `load_state_dict(best_state)` restores **the weights from the best epoch**.

Note: since the network uses only ReLU and linear layers, train/eval computations barely differ, but the `eval()` habit is essential once you use Dropout/BatchNorm.


In [ ]:
# %% 9. Train the MLP ourselves and keep the weights from the point where validation MSE was best.
def fit_mlp(width):  # Takes one hidden-layer width and returns the trained model and its evaluation records.
    torch.manual_seed(SEED)  # Use the same random seed so every candidate gets a comparable initialization.
    model = OneHiddenMLP(width).to(DEVICE)  # Build the model and move it to the CPU or GPU.
    x_train_tensor = torch.tensor(X_train, dtype=torch.float32)  # Convert the training inputs to a PyTorch float tensor.
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)  # Convert the training targets to a PyTorch float tensor.
    dataset = TensorDataset(x_train_tensor, y_train_tensor)  # Pair the input and target of the same row into one sample.
    loader = DataLoader(dataset, batch_size=MLP_BATCH_SIZE, shuffle=True)  # Build mini-batches and reshuffle the order every epoch.
    full_train_x = x_train_tensor.to(DEVICE)  # Prepare the inputs for computing the exact full train MSE at the end of each epoch.
    full_train_y = y_train_tensor.to(DEVICE)  # Prepare the targets for computing the full train MSE at the end of each epoch.
    val_x_tensor = torch.tensor(X_val, dtype=torch.float32, device=DEVICE)  # Create the validation inputs on the same compute device.
    val_y_tensor = torch.tensor(y_val, dtype=torch.float32, device=DEVICE)  # Create the validation targets on the same compute device.
    optimizer = torch.optim.Adam(model.parameters(), lr=MLP_LR)  # Adam is in charge of updating all learnable weights.
    loss_fn = nn.MSELoss()  # Training minimizes the mean of (prediction − target)^2.
    best_val_mse = float("inf")  # There is no validation score yet, so start the best score at infinity.
    best_epoch = 0  # Records the epoch number with the best validation performance.
    best_state = copy.deepcopy(model.state_dict())  # Copy of the weights to restore when a good result appears.
    epochs_without_improve = 0  # Number of consecutive epochs in which the validation loss did not improve.
    history = []  # Records the train/val MSE of every epoch, for visualizing the training process.
    for epoch in range(1, MLP_MAX_EPOCHS + 1):  # Repeat training from 1 up to the maximum number of epochs.
        model.train()  # Switch the network to training mode.
        for batch_x, batch_y in loader:  # Fetch one mini-batch of the training data.
            batch_x = batch_x.to(DEVICE)  # Move the input batch to the same device as the model.
            batch_y = batch_y.to(DEVICE)  # Move the target batch to the same device as well.
            optimizer.zero_grad()  # Clear the gradients left over from the previous batch.
            batch_pred = model(batch_x)  # Predict y for this batch with the network.
            batch_loss = loss_fn(batch_pred, batch_y)  # Compute the mean squared error of this batch.
            batch_loss.backward()  # Compute the gradients of all weights via the chain rule / backpropagation.
            optimizer.step()  # Adam updates the weights using the computed gradients.
        model.eval()  # Switch to evaluation mode for the validation computation.
        with torch.no_grad():  # We only evaluate here, so do not build an unnecessary autograd graph.
            epoch_train = loss_fn(model(full_train_x), full_train_y).item()  # Get the full training MSE of the current epoch as a number.
            epoch_val = loss_fn(model(val_x_tensor), val_y_tensor).item()  # Get the full validation MSE of the current epoch as a number.
        history.append({"epoch": epoch, "train_mse": epoch_train, "val_mse": epoch_val})  # Record one row of the training history.
        if epoch_val < best_val_mse - 1e-8:  # Check whether it improved sufficiently over the previous best validation MSE.
            best_val_mse = epoch_val  # Register this epoch as the best score.
            best_epoch = epoch  # Store the epoch number where the best score occurred.
            best_state = copy.deepcopy(model.state_dict())  # Copy and keep the model's entire best set of weights.
            epochs_without_improve = 0  # Performance improved, so reset the patience counter.
        else:  # This part runs if the validation MSE did not improve.
            epochs_without_improve += 1  # Increment the number of epochs without improvement by one.
        if epochs_without_improve >= MLP_PATIENCE:  # Check whether it has gone too long in a row without improving.
            break  # End the training loop to avoid overfitting and unnecessary computation.
    model.load_state_dict(best_state)  # Restore the weights with the 'lowest validation MSE', not those of the last epoch.
    model.eval()  # Keep the model in evaluation mode for final predictions.
    history_table = pd.DataFrame(history)  # Convert the per-epoch records to a DataFrame.
    final_train_mse = mse(y_train, predict_mlp(model, X_train))  # Compute the train MSE of the restored best weights.
    final_val_mse = mse(y_val, predict_mlp(model, X_val))  # Compute the val MSE of the restored best weights.
    return model, best_epoch, final_train_mse, final_val_mse, history_table  # Return the trained model together with all evaluation results.

### Code 10. Why does `predict_mlp` use `no_grad` and `.cpu().numpy()`?

When predicting, there is no need to modify the weights. So `torch.no_grad()` prevents building the autograd graph and the memory it consumes. Even if the model is on the GPU, CSV writing and NumPy metric computations usually expect CPU arrays, so we convert with `.detach().cpu().numpy()`. `chunk_size=4096` lets us predict even a 160×160=25,600-point grid **in the original order** without memory pressure.


In [ ]:
# %% 10. Write a function that predicts with the PyTorch model and converts back to a NumPy array.
def predict_mlp(model, X, chunk_size=4096):  # A function that can predict in pieces even when there is a lot of data.
    model.eval()  # Put layers whose behavior may differ between training and evaluation (e.g., Dropout) into evaluation mode.
    pieces = []  # Empty list to accumulate the results of the prediction batches.
    with torch.no_grad():  # Backpropagation is not needed for prediction, so turn off gradient tracking.
        for start in range(0, len(X), chunk_size):  # Process the input rows in pieces of chunk_size.
            batch = X[start:start + chunk_size]  # Take only the current prediction batch from the input array.
            tensor = torch.tensor(batch, dtype=torch.float32, device=DEVICE)  # Make the batch a tensor on the same device as the network.
            predicted = model(tensor).detach().cpu().numpy()  # Convert the prediction tensor to a CPU NumPy array.
            pieces.append(predicted)  # Keep this batch's results in order.
    return np.concatenate(pieces) if pieces else np.empty(0, dtype=np.float32)  # Join all predictions in the original row order.

### Code 11. What does the MLP hyperparameter change?

We change only the number of hidden nodes `H`, over `16,32,64,128`. The other conditions (Adam learning rate, batch size, maximum epochs, early-stopping criterion, seed) are fixed. We record the train/validation MSE computed at each candidate's best epoch, and select the `H` with the smallest validation value. A lower train MSE does not guarantee better predictions on other data, so **the selection criterion must be validation**.


In [ ]:
# %% 11. [3(c)] Vary the hidden-layer width and record the train/validation MSE in a table.
mlp_rows = []  # Create a list to store the evaluation results for each width.
for width in MLP_WIDTH_VALUES:  # Compare models with 16, 32, 64, 128 nodes in turn.
    candidate, epoch, train_mse, val_mse, history_table = fit_mlp(width)  # Train one model and get its early-stopping results.
    mlp_rows.append({"hidden_width": width, "best_epoch": epoch, "train_mse": train_mse, "val_mse": val_mse})  # Record the selection criteria.
    history_table.to_csv(OUTPUT_DIR / f"3c_mlp_history_width_{width}.csv", index=False)  # Keep the per-epoch learning curve for this width.
    print(f"MLP width={width:3d} | best epoch={epoch:3d} | train MSE={train_mse:.6f} | val MSE={val_mse:.6f}")  # Print the candidate's performance.
mlp_table = pd.DataFrame(mlp_rows)  # Turn the result list into a table.
mlp_table.to_csv(OUTPUT_DIR / "3c_mlp_sweep.csv", index=False)  # Save the results for all widths to CSV.
best_mlp_idx = int(mlp_table["val_mse"].idxmin())  # Find the row with the lowest validation loss.
best_width = int(mlp_table.loc[best_mlp_idx, "hidden_width"])  # Select the optimal width from that row.
print(f"Selected best MLP hidden width: {best_width}")  # Print the MLP's optimal setting.
plot_sweep(mlp_table, "hidden_width", "train_mse", "3(c) MLP training MSE vs width", "3c_mlp_training_mse.png")  # Save the train MSE vs width plot.
plot_sweep(mlp_table, "hidden_width", "val_mse", "3(c) MLP validation MSE vs width", "3c_mlp_validation_mse.png")  # Save the validation MSE vs width plot.

### Code 12. What does "refit with the optimal hyperparameters" mean?

Problem 3(d) asks us to **train the two final models anew and compare them** using the settings chosen in (b) and (c). Here we retrain from scratch with the best `D` and the best `H`. Validation is still reserved for early stopping/selection, so **we do not train on Train+Validation combined.** If we merged them, we could no longer report the same validation MSE as an independent comparison figure.


In [ ]:
# %% 12. [3(d)] Retrain both models with the selected settings and compare their validation losses.
final_rks = RandomKitchenSinks(D=best_D, sigma=RKS_SIGMA, ridge_lambda=RKS_LAMBDA, seed=SEED)  # Build a new RKS with the D chosen by validation.
final_rks.fit(X_train, y_train)  # Recompute the final RKS coefficients using only the training data.
final_mlp, final_epoch, mlp_train_mse, mlp_val_mse, final_history = fit_mlp(best_width)  # Retrain the MLP of the selected width from initialization.
rks_train_mse = mse(y_train, final_rks.predict(X_train))  # Compute the final RKS training MSE.
rks_val_mse = mse(y_val, final_rks.predict(X_val))  # Compute the final RKS validation MSE.
comparison = pd.DataFrame([  # Put the performance of both models into one table.
    {"model": "RKS", "selected_hyperparameter": f"D={best_D}", "train_mse": rks_train_mse, "val_mse": rks_val_mse},  # One row with the RKS results.
    {"model": "MLP", "selected_hyperparameter": f"hidden_width={best_width}", "train_mse": mlp_train_mse, "val_mse": mlp_val_mse},  # One row with the MLP results.
])  # End of DataFrame construction.
comparison.to_csv(OUTPUT_DIR / "3d_model_comparison.csv", index=False)  # Save the numeric table for the submission report as CSV.
final_history.to_csv(OUTPUT_DIR / "3d_final_mlp_learning_curve.csv", index=False)  # Also save the final MLP learning curve.
print("\n[3(d)] Final validation performance\n" + comparison.to_string(index=False))  # Print the comparison table to the terminal.

### Code 13. What can we do with `test.csv`, which has no answers?

`final_rks.predict(X_test)` and `predict_mlp(final_mlp,X_test)` both produce numeric arrays of length `(200,)`. `np.savetxt(..., fmt="%.10f")` writes **only one number per row, with no header**. Since `X_test` is never shuffled or sorted, **the row order of the original test.csv**, as the professor requires, is preserved. Because `test.csv` has no true y, we never compute a test MSE or write down made-up figures.


In [ ]:
# %% 13. Test has no answers, so we only produce predictions and export them to CSV in the original row order.
rks_test_pred = final_rks.predict(X_test)  # Predict each row of test.csv with the final RKS.
mlp_test_pred = predict_mlp(final_mlp, X_test)  # Predict the same test.csv rows with the final MLP.
assert len(rks_test_pred) == len(X_test)  # Check that the number of RKS predictions equals the number of test rows.
assert len(mlp_test_pred) == len(X_test)  # Check that the number of MLP predictions also matches the original.
assert np.isfinite(rks_test_pred).all() and np.isfinite(mlp_test_pred).all()  # Check for NaN or infinite predictions.
np.savetxt("rks_test_predictions.csv", rks_test_pred, fmt="%.10f", delimiter=",")  # Save the RKS submission CSV: no header, one predicted number per row.
np.savetxt("mlp_test_predictions.csv", mlp_test_pred, fmt="%.10f", delimiter=",")  # Save the MLP submission CSV: no header, one predicted number per row.
print("Test predictions saved: rks_test_predictions.csv / mlp_test_predictions.csv")  # Report the actual file names.

### Code 14. How does a prediction "surface" become a picture?

The model only tells us the height at one point `(x1,x2)`. To make it look like a map, we create 160 coordinates from -3 to 3 along each of the horizontal and vertical axes and predict every combination — **160×160=25,600 points**. `meshgrid` builds the grid, `ravel` flattens the 2D grid into 1D order, `column_stack` builds the `[[x1,x2],...]` form, and `reshape` turns the predictions back into the grid shape.


In [ ]:
# %% 14. [3(d)] Build a dense grid over all of [-3,3]×[-3,3] and draw both models' surfaces.
axis_values = np.linspace(-3.0, 3.0, 160)  # Create 160 evenly spaced coordinates from -3 to 3.
mesh_x1, mesh_x2 = np.meshgrid(axis_values, axis_values)  # Build every (x1,x2) grid combination.
grid_inputs = np.column_stack((mesh_x1.ravel(), mesh_x2.ravel()))  # Convert into a prediction input matrix of shape (160^2,2).
rks_grid_pred = final_rks.predict(grid_inputs).reshape(mesh_x1.shape)  # Rearrange the RKS predictions into the 160×160 grid shape.
mlp_grid_pred = predict_mlp(final_mlp, grid_inputs).reshape(mesh_x1.shape)  # Rearrange the MLP predictions into the same grid.
color_low = float(min(y_train.min(), rks_grid_pred.min(), mlp_grid_pred.min()))  # Unify the color-scale minimum across the three figures.
color_high = float(max(y_train.max(), rks_grid_pred.max(), mlp_grid_pred.max()))  # Unify the color-scale maximum across the three figures.

### Code 15. The two models' colors must be compared on the same scale

If the RKS plot and the MLP plot have different color scales, you get the illusion that one is "brighter". So we compute the common minimum/maximum across the three figures and use the same numbers for `vmin` and `vmax`. `contourf` fills regions with similar values with the same color. The results are `3d_rks_surface.png` and `3d_mlp_surface.png`.


In [ ]:
# %% 15. Save a separate figure of each prediction surface using the common color range.
def save_surface(Z, title, filename):  # Takes a prediction grid Z and a file name and saves a 2D filled-contour color map.
    fig, ax = plt.subplots(figsize=(7, 5.8))  # Create the figure and axes.
    contour = ax.contourf(mesh_x1, mesh_x2, Z, levels=35, cmap="viridis", vmin=color_low, vmax=color_high)  # Show the spatial distribution of the predictions on the grid as color.
    ax.set_xlabel("x1")  # Show the x1 axis name.
    ax.set_ylabel("x2")  # Show the x2 axis name.
    ax.set_title(title)  # Show in the title whether it is the RKS or the MLP surface.
    ax.set_xlim(-3, 3)  # Set the horizontal-axis range to match the problem's input range.
    ax.set_ylim(-3, 3)  # Set the vertical-axis range to match the problem's input range.
    ax.set_aspect("equal", adjustable="box")  # Use the same scale on the horizontal and vertical axes.
    fig.colorbar(contour, ax=ax, label="predicted y")  # Show the predicted y magnitude corresponding to each color.
    fig.tight_layout()  # Prevent labels from being cut off.
    fig.savefig(OUTPUT_DIR / filename, dpi=180)  # Save this prediction surface as a PNG.
    plt.close(fig)  # Close the figure object.

save_surface(rks_grid_pred, f"3(d) RKS fitted surface, D={best_D}", "3d_rks_surface.png")  # Save the final RKS prediction map.
save_surface(mlp_grid_pred, f"3(d) MLP fitted surface, width={best_width}", "3d_mlp_surface.png")  # Save the final MLP prediction map.

### Code 16. Judge by eye whether the real pattern was learned

We lay out the scatter plot of training y drawn in (a) and the two models' surfaces with the same size and color scale. First look at where the high and low regions are in the original data. Then compare how well the two models preserve **the locations of lows and highs and the smooth variation**. A good interpretation uses the validation MSE number and these figures **together**.


In [ ]:
# %% 16. Compare the training-data pattern and the RKS/MLP predictions side by side on the same scale.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharex=True, sharey=True)  # Create three panels: actual data, RKS, MLP, from left to right.
axes[0].scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="viridis", vmin=color_low, vmax=color_high, s=12)  # Show the actual colors of the training data.
axes[0].set_title("Observed training y")  # State what the first panel shows.
axes[1].contourf(mesh_x1, mesh_x2, rks_grid_pred, levels=35, cmap="viridis", vmin=color_low, vmax=color_high)  # Show the RKS predicted spatial pattern.
axes[1].set_title(f"RKS, D={best_D}")  # State the model and setting of the second panel.
axes[2].contourf(mesh_x1, mesh_x2, mlp_grid_pred, levels=35, cmap="viridis", vmin=color_low, vmax=color_high)  # Show the MLP predicted spatial pattern.
axes[2].set_title(f"MLP, width={best_width}")  # State the model and setting of the third panel.
for ax in axes:  # Apply the same display range to all three axes.
    ax.set_xlabel("x1")  # Use a consistent horizontal-axis name.
    ax.set_ylabel("x2")  # Use a consistent vertical-axis name.
    ax.set_xlim(-3, 3)  # Use the same x1 range.
    ax.set_ylim(-3, 3)  # Use the same x2 range.
    ax.set_aspect("equal", adjustable="box")  # Give the three panels the same spatial aspect ratio.
fig.tight_layout()  # Space the three plots appropriately.
fig.savefig(OUTPUT_DIR / "3d_three_panel_comparison.png", dpi=180)  # Save the comparison figure, the most important one in (d).
plt.close(fig)  # Clean up the memory of the figure we no longer need.

### Code 17. What "training error goes down while validation error goes up" means

As training goes on, the MLP may fit the training points better while getting the unseen validation points more wrong. This is one sign of **overfitting**. We draw both curves together and put a dashed line at `best_epoch`, where the validation MSE was best. The model has already been restored to the weights from this epoch.


In [ ]:
# %% 17. Record the final MLP's validation/training losses in the same figure to show how it was trained.
fig, ax = plt.subplots(figsize=(7, 4.5))  # Create the early-stopping learning-curve plot.
ax.plot(final_history["epoch"], final_history["train_mse"], label="Train MSE")  # Plot the training error per epoch.
ax.plot(final_history["epoch"], final_history["val_mse"], label="Validation MSE")  # Plot the validation error per epoch.
ax.axvline(final_epoch, linestyle="--", label=f"Best epoch={final_epoch}")  # Mark the epoch that achieved the best validation error.
ax.set_xlabel("Epoch")  # State the unit of the horizontal axis.
ax.set_ylabel("MSE")  # State the loss metric on the vertical axis.
ax.set_title("3(d) Final MLP learning curves")  # Add a title to the plot.
ax.legend()  # Show the names of the lines.
ax.grid(True, alpha=0.3)  # Add a faint grid so numeric trends are easier to see.
fig.tight_layout()  # Tidy up so the figure is not cut off.
fig.savefig(OUTPUT_DIR / "3d_mlp_learning_curve.png", dpi=180)  # Save, as a PNG, how training and validation diverge.
plt.close(fig)  # Close the figure object.

### Code 18. Automatically filling the submission text with numbers

We fill the text with the RKS and MLP `train_mse`, `val_mse`, and best hyperparameters, and save it to `3_submittable_english_draft.txt`. It states which model did better based on the validation values and adds an interpretation of the two surfaces seen in the three-panel figure. The test set has no answers, so no test performance figures are written.


In [ ]:
# %% 18. Auto-fill the English submission draft with the actual run's numbers (review the interpretation against the plots).
winner = "RKS" if rks_val_mse < mlp_val_mse else "MLP"  # Pick the name of the model with the smaller validation MSE.
english_summary = f"""Problem 3: Random Kitchen Sinks versus One-Hidden-Layer MLP

(a) Data visualization
We plotted the 600 labeled training examples as a 2D scatter plot, with x1 and x2 on the axes and observed y represented by color. See 3a_training_scatter.png.

(b) Random Kitchen Sinks with ridge regression
We transformed each two-dimensional input into D sinusoidal features: sin(w_j^T x+b_j) and cos(w_j^T x+b_j), where w_j ~ N(0,I_2/sigma^2), b_j ~ Uniform(0,2pi), h=D/2, sigma={RKS_SIGMA}, and the random seed was fixed at {SEED}. We fit an L2-regularized least-squares model with lambda={RKS_LAMBDA}, excluding the intercept from the penalty. We swept D over {RKS_D_VALUES} and chose D={best_D} by validation MSE. The refitted model achieved train MSE={rks_train_mse:.6f} and validation MSE={rks_val_mse:.6f}. See 3b_rks_training_mse.png and 3b_rks_validation_mse.png.

(c) One-hidden-layer MLP
We trained a neural network with two input units, one fully connected ReLU hidden layer, and one linear output. We swept hidden layer width over {MLP_WIDTH_VALUES}, fixing Adam learning rate at {MLP_LR}, batch size at {MLP_BATCH_SIZE}, and early-stopping patience at {MLP_PATIENCE}. Using the validation MSE criterion, we selected width={best_width}. See 3c_mlp_training_mse.png and 3c_mlp_validation_mse.png.

(d) Comparison and visualization
We refitted each selected model on the training split and evaluated both against the labeled validation split. RKS validation MSE={rks_val_mse:.6f}; MLP validation MSE={mlp_val_mse:.6f}. The lower validation MSE was achieved by {winner}. Both predictions on the unlabeled 200-row test split were exported, preserving the original row order, to rks_test_predictions.csv and mlp_test_predictions.csv (headerless, one scalar per row). See 3d_three_panel_comparison.png, 3d_rks_surface.png, and 3d_mlp_surface.png.

Interpretation: RKS uses a fixed, randomly constructed smooth sinusoidal basis and fits only the linear output weights with ridge regression; the MLP learns both its hidden features and its output weights through backpropagation. In the three-panel figure both surfaces recover the trough near the left-center and the higher values toward the top and bottom; the RKS surface is smoother, while the ReLU MLP surface is slightly more piecewise-linear. No test MSE is reported because test.csv has no ground-truth y.
"""  # test has no ground-truth y, so the string above reports only validation numbers.
(OUTPUT_DIR / "3_submittable_english_draft.txt").write_text(english_summary, encoding="utf-8")  # Save the English draft, filled with numbers, as a text file.
print("\nProblem 3 complete: check the problem3_outputs folder, the two test-prediction CSVs, and the English submission draft.")  # Announce that all work is finished.

### If you have run all the code, view the result figures inside the notebook too

The next cell loads the PNGs from `problem3_outputs/` and displays them. (The display tool is needed only in the notebook, so it is not in the `.py`.)

In [ ]:
from IPython.display import display, Image  # IPython tools for displaying PNG results directly in a notebook cell.
from pathlib import Path  # Builds the path of the folder containing the PNG files.
for image_name in ["3a_training_scatter.png", "3b_rks_training_mse.png", "3b_rks_validation_mse.png", "3c_mlp_training_mse.png", "3c_mlp_validation_mse.png", "3d_three_panel_comparison.png"]:  # View the six important figures in order.
    print(image_name)  # Show the name of the figure file being viewed.
    display(Image(filename=str(Path("problem3_outputs") / image_name), width=820))  # Display that PNG on screen.


---

# Part E. How to interpret the actual experimental results

## 13. Numbers from running on the professor's data

**RKS: D=20, σ=1, λ=0.01**

| D | Train MSE | Validation MSE |
|---:|---:|---:|
| 20 | 0.035776 | **0.036124** |
| 40 | 0.032861 | 0.036553 |
| 80 | 0.031879 | 0.036887 |
| 160 | 0.030390 | 0.038196 |
| 320 | 0.029573 | 0.041269 |

**MLP: LR=0.002, batch=64, max 600 epochs, patience=45**

| Hidden width | Best Epoch | Train MSE | Validation MSE |
|---:|---:|---:|---:|
| 16 | 524 | 0.037643 | 0.042134 |
| 32 | 279 | 0.036007 | 0.042949 |
| 64 | 407 | 0.030971 | 0.039303 |
| 128 | 204 | 0.031993 | **0.038674** |

**Final comparison**: RKS validation MSE **0.036124** vs MLP **0.038674**. With this setup and data, RKS is slightly better. However, repeated experiments would be needed to claim that this gap holds for every random seed or other settings.

### Why does training improve but validation get worse as D grows?

As the RKS D grows from 20→320, **the set of curves it can represent grows**, and the train MSE dropped from 0.035776→0.029573. Meanwhile, the validation MSE rose from 0.036124→0.041269. This is evidence of a **generalization gap**: many features can also fit chance fluctuations in the training data. Here `λ` was fixed, so as D changes, the *relative effect* of the regularization strength can change too.

### Why does the MLP fit the training data better but fall behind on validation data?

The best MLP's train MSE `0.031993` is lower than RKS's `0.035776`. Yet on validation the MLP is worse. In other words, **fitting the training data better is not the same as predicting new data well.** The MLP is flexible because it can also learn its hidden features, but that flexibility can show up in some cases as a larger train–validation gap. We do not draw the categorical conclusion that "the MLP always overfits".

### Details to interpret from the figures

In the three-panel figure, the actual observations show low values near the center-left and high values at the top and bottom. Both RKS and MLP reproduce this large-scale structure. The RKS surface is smooth overall, as befits a combination of sines/cosines, while the ReLU MLP shows more angular changes in some regions. The validation MSE gap here is small, so **even if they look visually similar**, one can still come out ahead numerically.

## 14. Checklist of the professor's requirements

- [x] 3(a): training scatter plot of `x1, x2` with `y` shown as color
- [x] 3(b): sin/cos RKS implemented from scratch + ridge + D sweep + train/validation MSE **each in a separate figure**
- [x] 3(c): PyTorch MLP with exactly one hidden layer + width sweep + train/validation MSE **each in a separate figure**
- [x] 3(d): refit with the best settings, report validation MSE, two prediction-surface figures, comparison with the original data
- [x] `rks_test_predictions.csv`, `mlp_test_predictions.csv`: **one number per row**, in the same order as the 200 test rows
- [x] Complete `.py` code, step-by-step `.ipynb`, English submission draft
- [x] Test has no answers, so no test MSE is reported

**Caution on academic interpretation**: this single split alone does not prove that the gap in validation performance is statistically significant. The values in the prediction files were computed from the actual provided `test.csv`, but since it has no answers, the final graded score is unknown.

## 15. Role of each file

| File | What it is used for |
|---|---|
| `code/problem3_rks_mlp.py` | Complete implementation for submission; Korean comments on every line of code |
| `notebooks_en/problem3_study.ipynb` | The study notebook you are reading now, from elementary-school level to expert (Korean original: `notebooks/`) |
| `problem3_outputs/` | (a)–(d) figures, sweep tables, English draft (`outputs/problem3/problem3_outputs/` when run via `run_all.py`) |
| `submissions/rks_test_predictions.csv`, `submissions/mlp_test_predictions.csv` | Submission files required by 3(d) |

### Things to double-check before submitting

Confirm that `python run_all.py --problem 3` (from the `homework_2` folder) **actually runs successfully**. Before submitting, check again that the plots and numbers match the interpretation in the report.

